# 🏛️ Módulo Magistral 3: Base de Datos Analítica & BigQuery AI / ML (`%%bqsql`)
> **Nivel:** Senior / Staff Data & ML Platform Engineer  
> **Stack:** Google BigQuery, Vertex AI Foundation Models (**TimesFM 2.0**, **Gemini 2.5 Flash**), Native SQL (`%%bqsql`)

---

## 🎯 Arquitectura de Base de Datos Analítica de Última Generación
En lugar de exportar terabytes de datos fuera del Data Warehouse hacia servidores externos para detectar anomalías, predecir demanda o clasificar texto, **BigQuery AI & ML ejecuta modelos fundacionales directamente dentro del motor SQL**:

1. **Exploración Estadística Obligatoria (`ML.DESCRIBE_DATA`):** Antes de cualquier modelo, inspeccionamos cuantiles, nulos, media y desviación estándar en una sola pasada distribuida.
2. **Detección de Anomalías Zero-Shot (`AI.DETECT_ANOMALIES`):** Impulsado por el modelo fundacional de series temporales **TimesFM 2.0** sin necesidad de entrenar un modelo manual.
3. **Forecasting y Evaluación Rigurosa (`AI.FORECAST` + `AI.EVALUATE`):** División cronológica estricta en datos históricos (`history_data`) y datos reales de validación (`actual_data`), calculando MAE, RMSE y sMAPE.
4. **Extracción Estructurada con IA Generativa (`AI.GENERATE_TABLE`):** Transforma descripciones de transacciones no estructuradas en columnas SQL fuertemente tipadas usando `gemini-2.5-flash`.

In [ ]:
%%bqsql
-- 1. Creación del esquema analítico y tabla de transacciones en streaming
CREATE SCHEMA IF NOT EXISTS `my_project.financial_lakehouse`;

-- Muestra rápida de las primeras 5 filas (Regla obligatoria de exploración previa)
SELECT
  transaction_id,
  user_id,
  amount,
  merchant_category,
  country,
  txn_timestamp
FROM
  `my_project.financial_lakehouse.streaming_transactions`
LIMIT 5;

-- Estadísticas descriptivas completas del dataset mediante ML.DESCRIBE_DATA
SELECT
  *
FROM
  ML.DESCRIBE_DATA(
    TABLE `my_project.financial_lakehouse.streaming_transactions`
  );


---
## 1. Detección de Anomalías (`AI.DETECT_ANOMALIES`) y Forecasting (`AI.FORECAST`) con TimesFM 2.0

### Reglas de Oro en BigQuery AI para Series Temporales
1. **Funciones con Valor de Tabla (TVFs):** Tanto `AI.DETECT_ANOMALIES`, `AI.FORECAST` como `AI.EVALUATE` son TVFs y **siempre deben ir en la cláusula `FROM`**.
2. **Operador de Argumentos Nombrados (`=>`):** Los parámetros como `data_col => 'total_volume'`, `timestamp_col => 'txn_date'` o `id_cols => ['merchant_category']` utilizan siempre la sintaxis `=>`.
3. **División Cronológica Estricta (TimesFM):** Para evaluar o detectar anomalías sin fuga de datos del futuro (*data leakage*), dividimos la serie temporal en:
   * **`history_data`:** Ventana histórica de referencia (ej. `txn_date < '2026-10-01'`).
   * **`actual_data` / `target_data`:** Ventana reciente sobre la que evaluamos la precisión o buscamos picos anómalos (`txn_date >= '2026-10-01'`).

In [ ]:
%%bqsql
-- 2. Detección de Anomalías Multivariante por Categoría con AI.DETECT_ANOMALIES (TimesFM 2.0)
WITH daily_category_volume AS (
  SELECT
    EXTRACT(DATE FROM txn_timestamp) AS txn_date,
    merchant_category,
    SUM(amount) AS total_volume
  FROM
    `my_project.financial_lakehouse.streaming_transactions`
  GROUP BY
    txn_date, merchant_category
)
SELECT
  merchant_category,
  time_series_timestamp,
  time_series_data,
  is_anomaly,
  lower_bound,
  upper_bound,
  anomaly_probability
FROM
  AI.DETECT_ANOMALIES(
    -- Contexto histórico de referencia
    (SELECT * FROM daily_category_volume WHERE txn_date < DATE('2026-10-01')),
    -- Ventana objetivo a inspeccionar
    (SELECT * FROM daily_category_volume WHERE txn_date >= DATE('2026-10-01')),
    data_col => 'total_volume',
    timestamp_col => 'txn_date',
    id_cols => ['merchant_category'],
    anomaly_prob_threshold => 0.95
  )
WHERE
  is_anomaly = TRUE
ORDER BY
  anomaly_probability DESC;


In [ ]:
%%bqsql
-- 3. Evaluación de Precisión (MAE, RMSE, sMAPE) y Predicción a 14 días con AI.EVALUATE y AI.FORECAST
WITH daily_category_volume AS (
  SELECT
    EXTRACT(DATE FROM txn_timestamp) AS txn_date,
    merchant_category,
    SUM(amount) AS total_volume
  FROM
    `my_project.financial_lakehouse.streaming_transactions`
  GROUP BY
    txn_date, merchant_category
)
-- Paso A: Evaluamos el error sobre la partición cronológica de validación (actual_data)
SELECT
  merchant_category,
  mean_absolute_error,
  root_mean_squared_error,
  symmetric_mean_absolute_percentage_error
FROM
  AI.EVALUATE(
    (SELECT * FROM daily_category_volume WHERE txn_date < DATE('2026-10-01')),
    (SELECT * FROM daily_category_volume WHERE txn_date >= DATE('2026-10-01')),
    data_col => 'total_volume',
    timestamp_col => 'txn_date',
    id_cols => ['merchant_category']
  );

-- Paso B: Predicción futura a 14 días con bandas de confianza del 95%
WITH daily_category_volume AS (
  SELECT
    EXTRACT(DATE FROM txn_timestamp) AS txn_date,
    merchant_category,
    SUM(amount) AS total_volume
  FROM
    `my_project.financial_lakehouse.streaming_transactions`
  GROUP BY
    txn_date, merchant_category
)
SELECT
  merchant_category,
  forecast_timestamp,
  forecast_value,
  prediction_interval_lower_bound,
  prediction_interval_upper_bound
FROM
  AI.FORECAST(
    (SELECT * FROM daily_category_volume),
    data_col => 'total_volume',
    timestamp_col => 'txn_date',
    id_cols => ['merchant_category'],
    horizon => 14,
    confidence_level => 0.95
  );


---
## 2. IA Generativa Estructurada en SQL (`AI.GENERATE_TABLE` con `gemini-2.5-flash`)

### Reglas Obligatorias de `AI.GENERATE_TABLE`
1. **Conexión Remota:** Usar `'DEFAULT'` en la cláusula `WITH CONNECTION` al declarar el modelo remoto con `ENDPOINT = 'gemini-2.5-flash'`.
2. **Alias `prompt`:** La subconsulta de entrada debe contener obligatoriamente una columna llamada o renombrada como `prompt`.
3. **Esquema Tipado entre Comillas:** `STRUCT("risk_level STRING, fraud_reason STRING, recommended_action STRING" AS output_schema)`.

In [ ]:
%%bqsql
-- 4. Creación del Modelo Remoto Gemini 2.5 Flash y Clasificación Estructurada con AI.GENERATE_TABLE
CREATE MODEL IF NOT EXISTS `my_project.financial_lakehouse.gemini_flash_model`
REMOTE WITH CONNECTION `DEFAULT`
OPTIONS (ENDPOINT = 'gemini-2.5-flash');

SELECT
  transaction_id,
  user_id,
  amount,
  risk_level,
  fraud_reason,
  recommended_action
FROM
  AI.GENERATE_TABLE(
    MODEL `my_project.financial_lakehouse.gemini_flash_model`,
    (
      SELECT
        transaction_id,
        user_id,
        amount,
        CONCAT(
          'Analiza esta transacción bancaria para auditoría antifraude: ',
          'Usuario=', user_id,
          ', Importe=€', CAST(amount AS STRING),
          ', Categoría=', merchant_category,
          ', País=', country
        ) AS prompt
      FROM
        `my_project.financial_lakehouse.streaming_transactions`
      WHERE
        amount > 2500
    ),
    STRUCT(
      "risk_level STRING, fraud_reason STRING, recommended_action STRING" AS output_schema,
      0.0 AS temperature
    )
  );
